## Modelo Toomer

1. *Núcleos Puntuales:* Representan toda la masa de cada galaxia, es decir, las interacciones gravitacionales se dan co y entre estos nucleos. 
2. *Estrelas como partículas de prueba:* La masa delas estrellas es $m_*=0$ y esto quiere decit que no interacutan entre ellas.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import jit
from typing import List, Dict, Any

In [2]:
class N_System:

    def __init__(self, particles: List[Dict[str, Any]]):
        "Clase general para sistemas de N partículas, donde cada partícula es un diccionario con sus propiedades."
        "mass: masa de la partícula, "
        "position: posición inicial (array de 3D), "
        "velocity: velocidad inicial (array de 3D)."

        self.n=len(particles)
        self.positions = np.array([p['position'] for p in particles])
        self.velocities = np.array([p['velocity'] for p in particles])
        self.masses = np.array([p['mass'] for p in particles])
        self.mass = self.masses.sum()

        v_prom=np.mean(np.linalg.norm(self.velocities, axis=1))

        #Tiempo de relajación estimado, usando la fórmula de Chandrasekhar
        self.t_relax = self.n / np.log(self.n) * (v_prom**3) / (self.mass * 1.0) 

        ##Indices de cuerpos con masa

        self.mass_indices = np.where(self.masses > 0)[0]




    def evolucionar(self, dt: float, T: float) -> np.ndarray[float, 3]:

        "Integra el sistema por el metodo de Leapfrog durante un tiempo T con pasos de dt"
        "params: dt: paso de tiempo, T: tiempo total de integración"
        "returns las posiciones de las partículas a lo largo del tiempo"
        "(array de shape (n_steps, n_particles, 3))"


        def aceleration(pos: np.ndarray,
                        masses: np.ndarray[float],
                        m_indices: np.ndarray[int],
                        n_particles: int
                        )-> np.ndarray[float, 3]:

            "Calcula la aceleración de cada partícula debido a la gravedad de las otras partículas"
            "params: positions: array de shape (n_particles, 3) con las posiciones actuales"
            "returns: array de shape (n_particles, 3) con las aceleraciones"

            acc = np.zeros_like((n_particles, 3))
            for i in range(self.n):
                for j in m_indices:
                    if i==j:
                        continue
                    r_vec = pos[i] - pos[j]
                    r_mag = np.linalg.norm(r_vec) + 1e-10  # Evitar división por cero
                    acc -= masses[j] * r_vec / r_mag**3

            return acc

        num_steps = int(T / dt)
        trajectories = np.zeros((num_steps, self.n, 3))

        #Leapfrog

        acceleration = acceleration(self.positions, self.masses, self.mass_indices, )
        self.velocities += 0.5 * acceleration * dt

        for step in range(1, num_steps):
            self.positions += self.velocities * dt
            trajectories[:, step,:] = self.positions
            acceleration = acceleration(self.positions, self.masses, self.mass_indices)
            self.velocities += acceleration * dt
        return trajectories


In [3]:
mi_primer_sistema = N_System([
    {'mass': 1.0, 'position': [0.0, 0.0, 0.0],
      'velocity': [0.0, 0.0, 0.0]},
    {'mass': 2.0, 'position': [1.0, 0.0, 0.0],
      'velocity': [0.0, 1.0, 0.0]},
    {'mass': 3.0, 'position': [0.0, 1.0, 0.0],
      'velocity': [0.0, 0.0, 1.0]}
])

In [4]:
mi_primer_sistema